In [1]:
import pandas as pd

In [2]:
tumor = "t57"
tumor_variants = pd.read_csv(f"../data/{tumor}/{tumor}.tumor_spec_variant_counts.txt", sep = "\t")

In [3]:
A_df = pd.read_csv(f"../data/{tumor}/{tumor}_population_assignments_A_WGS_bins.txt", sep="\t")

In [4]:
A_dict = {}
for cluster in A_df['population'].unique():
    print(cluster)
    cells_in_cluster = A_df[A_df['population'] == cluster]['cell'].tolist()
    A_dict[cluster] = {
        'cells': cells_in_cluster
    }

A111
A112
A1211
A1212
A1221
A1222
A2


In [5]:
tumor_variants = tumor_variants[tumor_variants["alt_count"]>0]

In [6]:
len(tumor_variants)

18593

In [7]:
tumor_variants["chrom"].unique()

array(['chr10', 'chr11', 'chr1', 'chr12', 'chr21', 'chr13', 'chr14',
       'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr2', 'chr20',
       'chr4', 'chr7', 'chr22', 'chr3', 'chr5', 'chr6', 'chr8', 'chr9',
       'chrX', 'chrY'], dtype=object)

In [8]:
chrom_set = ["chr"+str(i) for i in range(1,23)] + ["chrX"]

In [9]:
pop_var_dict = {}
tot_var_dict = {}
for pop in A_dict.keys():
    pop_var_dict[pop] = {}
    tot_var_dict[pop] = 0
for ch in chrom_set:
    ch_var = tumor_variants[tumor_variants['chrom'] == ch]
    for pop in A_dict.keys():
        pop_cells = A_dict[pop]['cells']
        pop_ch_var = ch_var[ch_var['cell_id'].isin(pop_cells)]
        cells_with_var = pop_ch_var['cell_id'].unique()
        pop_var = pop_ch_var["pos"].unique()
        pop_var_dict[pop][ch] = pop_var
        tot_var_dict[pop] += len(pop_var)
    #print(pop,len(cells_with_var), len(cells_with_var)/len(pop_cells))

In [10]:
# variants per cell in each cluster
for pop in tot_var_dict:
    print(pop, tot_var_dict[pop]/len(A_dict[pop]['cells']))

A111 8.232142857142858
A112 9.347222222222221
A1211 8.476
A1212 10.622641509433961
A1221 8.407407407407407
A1222 10.132530120481928
A2 0.7884057971014493


In [11]:
#pop_var_dict

In [12]:
tumor_variants = pd.read_csv(f"../data/{tumor}/{tumor}.tumor_spec_pseudo_variants.txt", sep = "\t")

In [13]:
tumor_variants = tumor_variants[tumor_variants["alt_count"]>0]

In [14]:
len(tumor_variants)

290

In [15]:
pop_var_dict = {}
tot_var_dict = {}
for pop in A_dict.keys():
    pop_var_dict[pop] = {}
    tot_var_dict[pop] = 0
for ch in chrom_set:
    ch_var = tumor_variants[tumor_variants['chrom'] == ch]
    for pop in A_dict.keys():
        pop_cells = A_dict[pop]['cells']
        pop_ch_var = ch_var[ch_var['cell_id'].isin(pop_cells)]
        cells_with_var = pop_ch_var['cell_id'].unique()
        pop_var = pop_ch_var["pos"].unique()
        pop_var_dict[pop][ch] = pop_var
        tot_var_dict[pop] += len(pop_var)
    #print(pop,len(cells_with_var), len(cells_with_var)/len(pop_cells))

In [16]:
# pseudo variants per cell in each cluster
for pop in tot_var_dict:
    print(pop, tot_var_dict[pop]/len(A_dict[pop]['cells']))

A111 0.0625
A112 0.06944444444444445
A1211 0.088
A1212 0.03773584905660377
A1221 0.05555555555555555
A1222 0.12048192771084337
A2 0.04173913043478261


In [17]:
# First, load the real variants
tumor_variants_real = pd.read_csv(f"../data/{tumor}/{tumor}.tumor_spec_variant_counts.txt", sep="\t")
tumor_variants_real = tumor_variants_real[tumor_variants_real["alt_count"] > 0]

# Then load pseudo variants
tumor_variants_pseudo = pd.read_csv(f"../data/{tumor}/{tumor}.tumor_spec_pseudo_variants.txt", sep="\t")
tumor_variants_pseudo = tumor_variants_pseudo[tumor_variants_pseudo["alt_count"] > 0]


In [19]:
# Initialize lists to store data
pop_names = []
num_variant_positions = []
num_cells_with_variants = []
variants_per_cell = []
num_pseudo_positions = []
num_cells_with_pseudo = []
pseudo_per_cell = []
# For each population, compute all metrics
for pop in A_dict.keys():
    pop_names.append(pop)
    pop_cells = A_dict[pop]['cells']
    num_cells = len(pop_cells)
    
    # Real variants
    pop_real_variants = tumor_variants_real[tumor_variants_real['cell_id'].isin(pop_cells)]
    num_variant_positions.append(len(pop_real_variants["pos"].unique()))
    num_cells_with_variants.append(len(pop_real_variants['cell_id'].unique()))
    variants_per_cell.append(len(pop_real_variants["pos"].unique()) / num_cells)
    
    # Pseudo variants
    pop_pseudo_variants = tumor_variants_pseudo[tumor_variants_pseudo['cell_id'].isin(pop_cells)]
    num_pseudo_positions.append(len(pop_pseudo_variants["pos"].unique()))
    num_cells_with_pseudo.append(len(pop_pseudo_variants['cell_id'].unique()))
    pseudo_per_cell.append(len(pop_pseudo_variants["pos"].unique()) / num_cells)


In [20]:
# Create the summary table
variant_summary = pd.DataFrame({
    'Population': pop_names,
    'Num_Cells': [len(A_dict[pop]['cells']) for pop in pop_names],
    'Variant_Positions': num_variant_positions,
    'Cells_With_Variants': num_cells_with_variants,
    'Variants_Per_Cell': variants_per_cell,
    'Pseudo_Variant_Positions': num_pseudo_positions,
    'Cells_With_Pseudo': num_cells_with_pseudo,
    'Pseudo_Per_Cell': pseudo_per_cell
})

In [22]:
variant_summary

,Population,Num_Cells,Variant_Positions,Cells_With_Variants,Variants_Per_Cell,Pseudo_Variant_Positions,Cells_With_Pseudo,Pseudo_Per_Cell
0,A111,224,1848,223,8.250000,14,16,0.062500
1,A112,144,1347,143,9.354167,10,12,0.069444
2,A1211,250,2123,249,8.492000,22,30,0.088000
3,A1212,53,565,52,10.660377,2,2,0.037736
4,A1221,54,455,54,8.425926,3,3,0.055556
5,A1222,83,843,83,10.156627,10,10,0.120482
6,A2,1725,1371,1356,0.794783,72,143,0.041739
